In [1]:
import openai

import json
import numpy as np
import os

from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

from huggingface_hub import InferenceClient

In [2]:
# Load the dataset
with open('ocpq_dataset.json', 'r') as f:
    dataset = json.load(f)

# Load the generated embeddings
embeddings = np.load('embeddings.npy')

# Load the local embedding model so we can embed user queries on the fly
model = SentenceTransformer('./local-all-MiniLM-L6-v2') 

def get_embedding(text):
    return model.encode(text)

# Initialize Hugging Face Inference Client for generation
hf_client = InferenceClient(api_key=os.getenv("HF_TOKEN"))

In [3]:
def retrieve_examples(query, dataset, embeddings, top_k=3):
    """
    Retrieve the top-k most similar examples from the dataset based on cosine similarity of embeddings.
    """
    query_emb = get_embedding(query)
    
    # Calculate cosine similarity between the query and all dataset questions
    similarities = cosine_similarity([query_emb], embeddings)[0]
    
    # Get the indices of the top-k most similar questions
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    
    examples = []
    for idx in top_indices:
        examples.append({
            "nl": dataset[idx]["nl"],
            "nl_clean_1": dataset[idx]["nl_clean_1"],
            "nl_clean_2": dataset[idx]["nl_clean_2"],
            "query": dataset[idx]["query"]
        })
    
    return examples

In [4]:
retrieve_examples("create order happens before ship order", dataset, embeddings)

[{'nl': 'The ship package event happens before the create order event.',
  'nl_clean_1': 'The ship package event occurs prior to the create order event.',
  'nl_clean_2': 'The ship package event takes place before the create order event occurs.',
  'query': {'nodes': [{'Box': [{'newEventVars': {'0': ['ship package'],
        '1': ['create order']},
       'newObjectVars': {'0': ['packages']},
       'filters': [],
       'sizeFilters': [],
       'constraints': [{'type': 'Filter',
         'filter': {'type': 'TimeBetweenEvents',
          'from_event': 0,
          'to_event': 1,
          'min_seconds': 0,
          'max_seconds': None}}],
       'evVarLabels': {},
       'obVarLabels': {}},
      []]}],
   'edgeNames': []}},
 {'nl': 'Whenever the ship package event occurs, the create order event eventually occurs.',
  'nl_clean_1': 'The ship package event triggers the create order event.',
  'nl_clean_2': 'Upon the ship package event, the create order event is triggered.',
  'query':

In [5]:
from LogEvents import events_and_objects_from_ocel

event_names, object_names = events_and_objects_from_ocel("../hospital_log_high.json")

Event names: ['Diagnosis', 'Discharge', 'Examination', 'Expert Examination', 'Register at ER', 'Register at FD', 'Thorough Examination', 'Treatment', 'Treatment successful', 'Treatment unsuccessful']
Object types: ['Person']


In [6]:
def nl_to_ocpq_rag(query, dataset, embeddings, query_types=None):
    if query_types is None:
        query_types = ["nl"]
        
    # Retrieve relevant examples
    examples = retrieve_examples(query, dataset, embeddings)
    
    # Construct a prompt with few-shot examples
    example_prompts = []
    for ex in examples:
        for qt in query_types:
            if qt in ex:
                example_prompts.append(
                    f"Question: {ex[qt]}\nJSON: {json.dumps(ex['query'])}"
                )
    few_shot_string = "\n\n".join(example_prompts)
    
    prompt = f"""
Translate natural language process queries into OCPQ JSON based on the examples and the current event and object names.
Output ONLY valid JSON.

EXAMPLES:
{few_shot_string}

EVENT NAMES:
{', '.join(event_names)}

OBJECT NAMES:
{', '.join(object_names)}

Now translate this query:
Question: {query}
JSON:
"""

    # Using a free open-source model via Hugging Face Serverless API
    response = hf_client.chat_completion(
        model="meta-llama/Meta-Llama-3-8B-Instruct",
        messages=[{"role":"user","content":prompt}],
        max_tokens=2000,
        response_format={"type": "json_object"}
    )

    content = response.choices[0].message.content
    
    try:
        # Llama-3 might add markdown quotes like ```json, so we strip them
        content = content.replace("```json", "").replace("```", "").strip()
        return json.loads(content)
    except json.JSONDecodeError:
        print("Failed to decode JSON. Raw output:")
        print(content)
        return None

In [7]:
nl_to_ocpq_rag("how can I check whether a person gets diagnosed before being registered at the ER?", dataset, embeddings)

{'nodes': [{'Box': [{'newEventVars': {'0': ['Diagnosis'],
      '1': ['Register at ER']},
     'newObjectVars': {'0': ['Person']},
     'filters': [{'type': 'O2E', 'object': 0, 'event': 1, 'qualifier': None},
      {'type': 'O2E', 'object': 0, 'event': 0, 'qualifier': None}],
     'sizeFilters': [],
     'constraints': [],
     'evVarLabels': {},
     'obVarLabels': {}},
    []]}],
 'edgeNames': []}

In [ ]:
nl_to_ocpq_rag("write me a poem about shipping order", dataset, embeddings)

{'nodes': [{'Box': [{'newEventVars': {'0': ['ship package']},
     'newObjectVars': {'0': ['poem']},
     'filters': [],
     'sizeFilters': [],
     'constraints': [],
     'evVarLabels': {},
     'obVarLabels': {}}]},
  []],
 'edgeNames': []}

In [ ]:
nl_to_ocpq_rag("no payment reminder should be sent out after payment has been made", dataset, embeddings)

{'nodes': [{'Box': [{'newEventVars': {'0': ['payment reminder sent out'],
      '1': ['payment has been made']},
     'newObjectVars': {'0': ['communications']},
     'filters': [],
     'sizeFilters': [],
     'constraints': [{'type': 'NOT', 'child_names': ['edge_1']}],
     'evVarLabels': {},
     'obVarLabels': {}}]},
  []],
 'edgeNames': []}

In [ ]:
def nl_to_ocpq(query):

    prompt = f"""
Translate natural language process queries into OCPQ JSON.
Output ONLY valid JSON.

Question: {query}
JSON:
"""

    response = hf_client.chat_completion(
        model="meta-llama/Meta-Llama-3-8B-Instruct",
        messages=[{"role":"user","content":prompt}],
        max_tokens=800
    )

    content = response.choices[0].message.content
    
    try:
        content = content.replace("```json", "").replace("```", "").strip()
        return json.loads(content)
    except json.JSONDecodeError:
        print("Failed to decode JSON. Raw output:")
        print(content)
        return None